# Base Data

In [12]:
# =============================================================================
# Query all tables from database: row count, min/max open_time
# =============================================================================

import sys
sys.path.insert(0, "..")
import utils
import sqlite3
import pandas as pd

# =============================================================================
# Load config
# =============================================================================
config = utils.load_db_config()
config["database"]["features"] = utils.load_features_config()["database"]["features"]
config["model"] = utils.load_models_config()["model"]
config["api"] = utils.load_env_config().get("api", {})
DB_PATH           = config["database"]["db_path"]
table_predictions = "bchusdt_1m_predictions"


In [ ]:
# Target column name (from config)
feat_cfg = utils.load_features_config()
targets = feat_cfg["database"]["features"]["targets"]
long_cfg = next((t for t in targets if t.get("direction") == "long"), None)
if not long_cfg:
    raise ValueError("No long target config found")
target_name = long_cfg.get("name") or utils.target_col_name("long", long_cfg["rolling_window"], long_cfg["percentile"])



In [14]:
# =============================================================================
# parameters
# =============================================================================
open_time_from  = "2017-01-01 00:00:00"
open_time_to    = "2025-10-31 23:59:00"
ROLLING_WINDOW  = 240

# =============================================================================
# Database connection and query
# =============================================================================
conn = sqlite3.connect(DB_PATH)

query = f"""
	SELECT *
	FROM {table_predictions}
"""
df_imp_a = pd.read_sql_query(query, conn)
conn.close()

df_imp_a.tail()


,open_time,close,target,feat_rsi_14,feat_roc_14,feat_roc_140,feat_sma_ratio_14,feat_sma_ratio_140,feat_bb_width_14,feat_bb_width_140,pred_prob
3135716,2025-11-15 15:40:00,506.8,0,42.721628,-0.157604,-1.015625,0.999422,0.994553,0.004171,0.018423,0.077275
3135717,2025-11-15 15:41:00,508.3,0,58.098542,0.137904,-0.896861,1.002282,0.997561,0.004738,0.018253,0.083431
3135718,2025-11-15 15:42:00,507.6,0,51.191840,-0.039386,-1.052632,1.000930,0.996262,0.004637,0.018176,0.081284
3135719,2025-11-15 15:43:00,507.5,0,50.272403,-0.078756,-1.129944,1.000789,0.996147,0.004409,0.018049,0.079857
3135720,2025-11-15 15:44:00,507.8,0,52.999672,-0.019689,-1.187001,1.001395,0.996821,0.004329,0.017826,0.079721


# Calculation of Actions

## Profit/Loss Rules

* **If price reaches +3% first** → Realize 3% profit
* **If price reaches -1% first** → Realize 1% loss
* **If price reaches +1% first** → Wait and observe:
    * If price then reaches +3% (overall from entry) → Realize 3% profit
    * If price then drops to -1% (overall from entry) → Realize 0% (breakeven)

# Summary

In [25]:
# =============================================================================
# summary
# =============================================================================
import pandas as pd

summary = pd.DataFrame({
    target_name: [0, 1, 'Total'],
    'db': [
        (df_imp_a[target_name] == 0).sum(),
        (df_imp_a[target_name] == 1).sum(),
        len(df_imp_a)
    ]
})

# Add ratio column (4 decimals)
summary['ratio'] = [
    round((df_imp_a[target_name] == 0).sum() / len(df_imp_a), 4),
    round((df_imp_a[target_name] == 1).sum() / len(df_imp_a), 4),
    1.0
]

# Add average pred_prob column
summary['avg_pred_prob'] = [
    round(df_imp_a[df_imp_a[target_name] == 0]['pred_prob'].mean(), 4),
    round(df_imp_a[df_imp_a[target_name] == 1]['pred_prob'].mean(), 4),
    round(df_imp_a['pred_prob'].mean(), 4)
]

# Format: add thousand separator to db column
summary['db'] = summary['db'].apply(lambda x: f"{x:,}")

display(summary)


,target,db,ratio,avg_pred_prob
0,0,"2,822,149",0.9,0.0931
1,1,"313,572",0.1,0.1618
2,Total,"3,135,721",1.0,0.1000
